In [1]:
from utilities_2 import preprocess, create_model_init, get_compute_metrics, concatenate_splits, fine_tuning_training_on_single_corpus
from huggingface_hub import HfApi
import os
import matplotlib as plt
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)
import numpy as np
import pandas as pd
from seqeval.metrics import f1_score
from transformers import EvalPrediction

from transformers import TrainingArguments
from datasets import get_dataset_config_names
from datasets import load_dataset

from transformers import DataCollatorForTokenClassification

from collections import defaultdict

from collections import Counter

from transformers import Trainer

import torch

from seqeval.metrics import classification_report

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA is available. Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA not available. Using CPU.")

CUDA is available. Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## Test small Ukranian models

In [3]:
model_name="xlm-roberta-large"
language_code="uk"

In [4]:
try:
    data = preprocess(language_code=language_code, model_name=model_name, train="all")
    tokenized_dataset, label_list, label2id, id2label, tokenizer = data[0], data[1], data[2], data[3], data[4]
    print("Sample from training data:")
    print(tokenized_dataset["train"][0])
    num_labels = len(label_list)
except NameError:
    print("Error: The 'preprocess' function is not defined.")
    print("Please ensure 'utilities.py' is in the same directory or accessible in your Python path,")
    print("and that it contains the 'preprocess' function.")
    # Example placeholder data if preprocess fails - replace with actual loading if needed
    tokenized_dataset = None # Set to None or load dummy data
    label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"] # Example labels
    label2id = {label: i for i, label in enumerate(label_list)}
    id2label = {i: label for i, label in enumerate(label_list)}
    num_labels = len(label_list)
    tokenizer = AutoTokenizer.from_pretrained(model_name) # Load tokenizer separately if needed
    print("\nWARNING: Using placeholder data because 'preprocess' failed.")

Map:   0%|          | 0/2211 [00:00<?, ? examples/s]

Map:   0%|          | 0/1850 [00:00<?, ? examples/s]

Map:   0%|          | 0/1493 [00:00<?, ? examples/s]

Sample from training data:
{'input_ids': [0, 174222, 229107, 148645, 55782, 92293, 415, 21726, 3494, 829, 22011, 591, 440, 1882, 41099, 6, 5, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [-100, 1, 10, 10, -100, 10, -100, 10, -100, 10, -100, 10, 10, -100, -100, 10, -100, -100, -100, -100, -100, -1

In [5]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2211
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1850
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1493
    })
})

In [5]:
model_init=create_model_init(model_name=model_name, data=data, device=device)
compute_metrics=get_compute_metrics(id2label=id2label)
data_collator = DataCollatorForTokenClassification(tokenizer)

In [6]:
training_args = TrainingArguments(
    output_dir="dump_this", log_level="error", num_train_epochs=3, 
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    save_steps=1e6, weight_decay=0.01, disable_tqdm=False, push_to_hub=False)

In [7]:
metrics_df = pd.DataFrame(columns=["num_samples", "f1_score"])
results = []
for num_samples in [250, 500, 1000, 2000]:
    result = fine_tuning_training_on_single_corpus(
        dataset=tokenized_dataset,
        num_samples=num_samples,
        model_init=model_init,
        training_args=training_args,
        data_collator=data_collator,
        xlmr_tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )
    results.append(result)

metrics_df = pd.concat([metrics_df, pd.DataFrame(results)], ignore_index=True)

c:\Users\user\Desktop\school\ITU\Sem4\NLPDL\NLPproject\NLP_project\scripts\development\utilities_2.py:416: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model_init=model_init, args=training_args,


Step,Training Loss


c:\Users\user\anaconda3\envs\nlp_gpu\lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\user\Desktop\school\ITU\Sem4\NLPDL\NLPproject\NLP_project\scripts\development\utilities_2.py:416: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model_init=model_init, args=training_args,


Step,Training Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 978.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 8.34 GiB is allocated by PyTorch, and 1.90 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
bg_score={"uk": 0.7736860922416875, "sl": 0.892269436114971}
ru_score={"uk": 0.8250615980288631, "sl": 0.8609369236444115}
#bg_ru_score=

In [ ]:
fig, ax = plt.subplots()
ax.axhline(bg_score["uk"], ls="--", color="r")
ax.axhline(ru_score["uk"], ls="--", color="g")
#ax.axhline(bg_ru_score["uk"], ls="--", color="c")
metrics_df.set_index("num_samples").plot(ax=ax)
plt.legend(["Zero-shot from bg", "Zero-shot from ru", "Fine-tuned on uk"], loc="lower right")
plt.ylim((0, 1))
plt.xlabel("Number of Training Samples")
plt.ylabel("F1 Score")
plt.show()